# UFO-120 Exploration (TURBIDITY CALIBRATION)
**Assigned to:** ___
**No GPU needed.**

Used to CALIBRATE our Jaffe-McGlamery augmentation and for qualitative validation.
**Has NO bounding box labels** — cannot compute mAP.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DATASET = "/content/drive/MyDrive/underwater_datasets/UFO-120"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "ufo" in n.lower(): DATASET = os.path.join(base, n); break
assert os.path.exists(DATASET), f"Not found"
print(f"✓ {DATASET}")


In [ ]:
from pathlib import Path; from collections import Counter
root = Path(DATASET)
for dp, dn, fn in sorted(os.walk(root)):
    depth = str(dp).replace(str(root),"").count(os.sep)
    if depth > 3: continue
    indent = "  "*depth
    exts = Counter(Path(f).suffix.lower() for f in fn)
    if exts:
        print(f"{indent}{os.path.basename(dp)}/")
        for e,c in sorted(exts.items()): print(f"{indent}  {c:>6} {e}")
    else:
        print(f"{indent}{os.path.basename(dp)}/")


In [ ]:
# === FIND CLEAN-TURBID PAIRS ===
from pathlib import Path; from collections import defaultdict
root = Path(DATASET)
folders = defaultdict(list)
for img in root.rglob("*"):
    if img.suffix.lower() in {'.jpg','.png','.bmp','.jpeg'}:
        folders[str(img.parent.relative_to(root))].append(img)

print("Image folders:")
for f in sorted(folders): print(f"  {f}: {len(folders[f])} images")
print("\nIdentify which folders are CLEAN vs TURBID pairs for calibration.")


In [ ]:
# === SAMPLE IMAGES FROM EACH FOLDER ===
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path; from collections import defaultdict
root = Path(DATASET)
folders = defaultdict(list)
for img in root.rglob("*"):
    if img.suffix.lower() in {'.jpg','.png','.bmp','.jpeg'}:
        folders[str(img.parent.relative_to(root))].append(img)

flist = sorted(folders.keys())[:8]
n = len(flist)
if n == 0: print("No images!")
else:
    fig,axes = plt.subplots(n,2, figsize=(12,3*n))
    if n==1: axes = axes.reshape(1,-1)
    for r,f in enumerate(flist):
        for c in range(min(2,len(folders[f]))):
            ax = axes[r][c]
            img = cv2.imread(str(folders[f][c]))
            if img is not None:
                ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
                h,w = img.shape[:2]
                ax.set_title(f"{f}\n({w}x{h})", fontsize=8)
            ax.axis("off")
    plt.suptitle("UFO-120 Samples", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.savefig("/content/ufo120_samples.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# === PSNR/SSIM (uncomment after identifying pairs) ===
# !pip install -q scikit-image
# from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
# import cv2, numpy as np
# from pathlib import Path
# root = Path(DATASET)
#
# CLEAN = root / "___"    # ← UPDATE
# TURBID = root / "___"   # ← UPDATE
#
# psnr_v, ssim_v = [], []
# for cf in sorted(CLEAN.glob("*"))[:50]:
#     tf = TURBID / cf.name
#     if not tf.exists(): continue
#     c = cv2.imread(str(cf)); t = cv2.imread(str(tf))
#     if c is None or t is None: continue
#     if c.shape != t.shape: t = cv2.resize(t, (c.shape[1],c.shape[0]))
#     psnr_v.append(psnr(c,t)); ssim_v.append(ssim(c,t,channel_axis=2))
#
# print(f"PSNR: {np.mean(psnr_v):.2f} ± {np.std(psnr_v):.2f}")
# print(f"SSIM: {np.mean(ssim_v):.4f} ± {np.std(ssim_v):.4f}")
print("Uncomment after identifying clean/turbid folder pairs above")


In [ ]:
print('''
================================================================
UFO-120 SUMMARY
================================================================
1. Total images: ___
2. Clean-turbid pairs exist? ___
3. Clean folder: ___
4. Turbid folder: ___
5. PSNR: ___ ± ___
6. SSIM: ___ ± ___
7. Has bbox annotations: NO
8. Usable for mAP: NO
9. Usable for calibration: ___
10. Usable for qualitative: ___
================================================================
''')
